## Regression Evaluation: Retinal Age Gap

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import json
from pathlib import Path

# Run in parent dir
cwd = Path.cwd().resolve()
if cwd.name == "regression_multitask":
    os.chdir(cwd.parent)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import mean_absolute_error, r2_score

from models import models
from utils import helpers, nako

### Parameters

In [ ]:
method = 'finetune'
backbone = 'vit'
img_size= (224, 224)
normalize = 'none'
kfold = 1
experiment_name = f'regmulti_{backbone}_{img_size[0]}_{method}_{normalize}_{kfold}'

device = 'cuda:0'
feature_name = ['a_rr_sys', 'a_rr_dia']
bins_sys = range(70, 250, 10)
bins_dias = range(40, 150, 10)

# Where results are saved
project_dir = Path.cwd()
models_dir = project_dir.joinpath('checkpoints')
model_file = models_dir.joinpath(f'{experiment_name}.pt')
results_file = models_dir.joinpath(f'{experiment_name}.json')

plots_dir = project_dir.joinpath('plots')
plots_dir.mkdir(parents=True, exist_ok=True)

### Load Results

In [ ]:
with open(results_file, 'r') as f:
       results = json.load(f)

loss_train, loss_val, ids, types, targets, preds = [np.array(results[k]) for k in ['loss_train', 'loss_val', 'ID', 'type', 'targets', 'preds']]
del results

### Learning curve

In [ ]:
fig, ax = plt.subplots(1, loss_train.ndim, figsize=(6.4*loss_train.ndim, 4.8))

for i in range(loss_train.shape[1]):
    ax[i].plot(range(loss_train.shape[0]), loss_train[:, i], '-o')
    ax[i].plot(range(loss_val.shape[0]), loss_val[:, i], '-o')
    ax[i].set_title('Loss')
    ax[i].set_xlabel('Epoch')
    ax[i].set_ylabel('Average Loss')
    ax[i].legend(['Train', 'Val'])

ax[0].set_ylim([0, 2 * loss_val[:, 0].mean()])

plt.tight_layout()

### Load Dataset

In [ ]:
nako_paths = nako.get_nako_paths(dataset='590', image_res=str(img_size[0]))
transform = models.get_augmentations(img_size, normalization=nako.NORMALIZATION)['test']

dataset_train = nako.NakoDataset(nako_paths, nako.IMAGE_TYPES, transform=transform, split='train', kfold=kfold, feature_name=feature_name, drop_nan=True)
dataset_test = nako.NakoDataset(nako_paths, nako.IMAGE_TYPES, transform=transform, split='test', kfold=kfold, feature_name=feature_name, drop_nan=True)

### Distribution of train and test set

In [ ]:
# Training dataset: to check distribution
df_train = pd.DataFrame({'ID': dataset_train.ids, 'targets_sys': dataset_train.labels[:, 0], 'targets_dias': dataset_train.labels[:, 1]})
df_train = df_train.astype({'ID': int})

# Test dataset
df_test = pd.DataFrame({'ID': ids, 'type': types, 
                        'targets_sys': targets[:, 0], 'preds_sys': preds[:, 0], 
                        'preds_dias': preds[:, 1], 'targets_dias': targets[:, 1]})
df_test = df_test.astype({'ID': int})

for target, bins in zip(['sys', 'dias'], [bins_sys, bins_dias]):
    df_test[f'error_signed_{target}'] = df_test[f'preds_{target}'] - df_test[f'targets_{target}']
    df_test[f'error_{target}'] = np.abs(df_test[f'error_signed_{target}'])
    df_test[f'bp_class_{target}'] = np.digitize(df_test[f'targets_{target}'], bins=bins) - 1
    df_test[f'bp_labels_{target}'] = df_test[f'bp_class_{target}'].map(lambda x: bins[x])

#### Distribution of age values for train and test set

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax = ax.flatten()

sns.histplot(df_train, x='targets_sys', bins=bins_sys, stat='percent', alpha=0.5, edgecolor='black', label='Train set', ax=ax[0])
sns.histplot(df_test, x='targets_sys', bins=bins_sys, stat='percent', alpha=0.5, edgecolor='black', label='Test set', ax=ax[0])
ax[0].legend()

sns.histplot(df_train, x='targets_dias', bins=bins_dias, stat='percent', alpha=0.5, edgecolor='black', label='Train set', ax=ax[1])
sns.histplot(df_test, x='targets_dias', bins=bins_dias, stat='percent', alpha=0.5, edgecolor='black', label='Test set', ax=ax[1])
ax[1].legend()

ax[0].set_xlabel('Systolic blood pressure')
ax[1].set_xlabel('Diastolic blood pressure')
ax[0].set_ylabel('Percentage of participants')
ax[0].set_title('SBP distribution')
ax[1].set_title('DBP distribution')
plt.tight_layout()

#### Mean Absolute Error

In [ ]:
mae_val_sys = mean_absolute_error(targets[:, 0], preds[:, 0])
print(f'Mean Absolute Error systolic (val) = {mae_val_sys:.2f}')
r2_val_sys = r2_score(targets[:, 0], preds[:, 0])
print(f'R2 systolic (val) = {r2_val_sys:.2f}')

mae_val_dias = mean_absolute_error(targets[:, 1], preds[:, 1])
print(f'Mean Absolute Error diastolic (val) = {mae_val_dias:.2f}')
r2_val_dias = r2_score(targets[:, 1], preds[:, 1])
print(f'R2 diastolic (val) = {r2_val_dias:.2f}')

#### Distribution of error by value

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 9))
ax = ax.flatten()

df_test['targets_sys_noise'] = df_test['targets_sys'] + (0.5 * np.random.rand(df_test.shape[0]))
sns.scatterplot(x='targets_sys_noise', y='preds_sys', data=df_test, alpha=0.8, s=5, marker='.', linewidths=None, edgecolors='None', ax=ax[0])
sns.lineplot(x=[0, 200], y=[0, 200], color='grey', linewidth=0.8, legend=False, ax=ax[0])
ax[0].set_xlabel('Real value')
ax[0].set_xlim([80, 200])
ax[0].set_ylim([80, 200])
ax[0].grid()
# ax[0].set_aspect('equal', 'box')
ax[0].set_ylabel('Predicted value')
ax[0].set_title('SBP predicted vs real value')

sns.boxplot(x='bp_labels_sys', y='error_signed_sys', data=df_test, fill=False, ax=ax[1]) # showfliers=False
ax[1].set_ylim([-100, 100])
ax[1].axhspan(ax[1].get_ylim()[0], 0, *ax[1].get_xlim(), color='lightgray', alpha=0.5) # Add grey shading below y=0
ax[1].set_ylabel('BP residuals')
ax[1].set_xlabel('Systolic blood pressure')
ax[1].set_title('SBP residuals')

df_test['targets_dias_noise'] = df_test['targets_dias'] + (0.5 * np.random.rand(df_test.shape[0]))
sns.scatterplot(x='targets_dias_noise', y='preds_dias', data=df_test, alpha=0.8, s=5, marker='.', linewidths=None, edgecolors='None', ax=ax[2])
sns.lineplot(x=[0, 200], y=[0, 200], color='grey', linewidth=0.8, legend=False, ax=ax[2])
ax[2].set_xlabel('Real value')
ax[2].set_xlim([40, 120])
ax[2].set_ylim([40, 120])
ax[2].grid()
ax[2].set_ylabel('Predicted value')
ax[2].set_title('DBP predicted vs real value')

sns.boxplot(x='bp_labels_dias', y='error_signed_dias', data=df_test, fill=False, ax=ax[3]) # showfliers=False
ax[3].set_ylim([-100, 100])
ax[3].axhspan(ax[1].get_ylim()[0], 0, *ax[1].get_xlim(), color='lightgray', alpha=0.5) # Add grey shading below y=0
ax[3].set_ylabel('BP residuals')
ax[3].set_xlabel('Diastolic blood pressure')
ax[3].set_title('DBP residuals')

plt.tight_layout()

### Aggregate K-fold results

In [ ]:
experiment_names = [f'regmulti_vit_224_finetune_none_{s}' for s in range(1, 6)]

mae_sys, mae_dias, r2_dias, r2_sys = [], [], [], []
for experiment_name in experiment_names:
    results_file = models_dir.joinpath(f'{experiment_name}.json')
    with open(results_file, 'r') as f:
        results = json.load(f)

    targets, preds = [np.array(results[k]) for k in ['targets', 'preds']]
    del results

    mae_sys.append(mean_absolute_error(targets[:, 0], preds[:, 0]))
    r2_sys.append(r2_score(targets[:, 0], preds[:, 0]))

    mae_dias.append(mean_absolute_error(targets[:, 1], preds[:, 1]))
    r2_dias.append(r2_score(targets[:, 1], preds[:, 1]))

df_kfold = pd.DataFrame({'MAE Systolic': mae_sys, 'R2 Systolic': r2_sys,
                            'MAE Diastolic': mae_dias, 'R2 Diastolic': r2_dias})

df_kfold

In [ ]:
df_kfold.describe()

### Compute performance estimates
Using bootstraping on the test set.

In [ ]:
experiment_names = ['regmulti_swin_224_finetune_none_4']

for experiment_name in experiment_names:
    print(experiment_name.split(sep='_')[1])
    results_file = models_dir.joinpath(f'{experiment_name}.json')

    with open(results_file, 'r') as f:
        results = json.load(f)

    targets, preds = [np.array(results[k]) for k in ['targets', 'preds']]

    print('Systolic')
    helpers.bootstrap_test(preds[:, 0], targets[:, 0], [mean_absolute_error, r2_score])
    print('Diastolic')
    helpers.bootstrap_test(preds[:, 1], targets[:, 1], [mean_absolute_error, r2_score])